In [1]:
import os
import boto3
import joblib
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

In [2]:
iris = load_iris()
X, y = iris.data, iris.target

In [3]:
model = DecisionTreeClassifier()
model.fit(X,y)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [4]:
joblib.dump(model, "iris-model.pkl")
print("Model saved as iris-model.pkl")

Model saved as iris-model.pkl


In [5]:
s3 = boto3.client("s3")
bucket = "vimal-jupy"
s3.upload_file("iris-model.pkl",bucket,"model-artifacts/iris-model.pkl")

In [6]:
print("Uploaded to S3 : ",f"s3://{bucket}/model-artifacts/iris-model.pkl")

Uploaded to S3 :  s3://vimal-jupy/model-artifacts/iris-model.pkl


In [7]:
import tarfile

with tarfile.open("model.tar.gz","w:gz") as tar:
    tar.add("iris-model.pkl")
    tar.add("inference.py")
print("model.tar.gz created")

model.tar.gz created


In [ ]:
import tarfile

with tarfile.open("model.tar.gz","w:gz") as tar:
    tar.add("iris-model.pkl")
    tar.add("inference.py")
print("tar file created")

In [8]:
s3.upload_file("model.tar.gz",bucket,"model-artifacts/model.tar.gz")

In [1]:
import sklearn
print(sklearn.__version__)

1.7.2


In [ ]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data="s3://vimal-jupy/model-artifacts/model.tar.gz",
    role="arn:aws:iam::943755222667:role/sagemaker-sudo-access",
    entry_point="inference.py",
    framework_version="0.23-1"
)

predictor = model.deploy(
    instance_type="ml.t2.medium",
    initial_instance_count=1
)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱  1 from sagemaker.sklearn.model import SKLearnModel                                            │
│    2                                                                                             │
│    3 model = SKLearnModel(                                                                       │
│    4 │   model_data="s3://vimal-jupy/model-artifacts/model.tar.gz",                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/deprecations.py:318 in find_spec          │
│                                                                                                  │
│   315 │   │   if leaf in _V3_REPLACEMENTS:                                                       │
│   316 │   │   │   # Curated, high-traffic module -> precise guidance.                            │
│   317 │   │   │   replacement, v3_import, docs_module = _V3_REPLACEMENTS[leaf]                   │
│ ❱ 318 │   │   │   raise_removed_in_v3(                                                           │
│   319 │   │   │   │   module=fullname,                                                           │
│   320 │   │   │   │   replacement=replacement,                                                   │
│   321 │   │   │   │   v3_import=v3_import,                                                       │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/deprecations.py:279 in                    │
│ raise_removed_in_v3                                                                              │
│                                                                                                  │
│   276 │   # leave a breadcrumb for log-captured environments without duplicating the             │
│   277 │   # message at WARNING level.                                                            │
│   278 │   logger.debug(msg)                                                                      │
│ ❱ 279 │   raise ModuleNotFoundError(msg, name=module)                                            │
│   280                                                                                            │
│   281                                                                                            │
│   282 class _RemovedV2ModuleFinder(importlib.abc.MetaPathFinder):                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: `sagemaker.sklearn` was removed in the SageMaker Python SDK v3. Use `ModelTrainer`. (from 
sagemaker.train import ModelTrainer)
Docs: https://sagemaker.readthedocs.io/en/stable/api/generated/sagemaker.train.model_trainer.html
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide.

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.training.configs import SourceCode
from sagemaker.serve import ModelBuilder


image_uri = image_uris.retrieve(
    framework="sklearn",
    region="ap-south-1",   
    version="1.9-0",
    instance_type="ml.t2.medium",
    image_scope="inference",
)
print(image_uri)

[09/23/26 08:49:35] INFO     Defaulting to only available Python version: py3                     ]8;id=8512789;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=8512790;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#624\624]8;;\

720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.9-0-cpu-py3


In [4]:
# !pip install -U sagemaker

In [ ]:
inference_source_code = SourceCode(
    source_dir=".",         # local folder containing inference.py
    entry_script="inference.py",
)

model_builder = ModelBuilder(
    image_uri=image_uri,
    source_code=inference_source_code,
    s3_model_data_url="s3://vimal-jupy/model-artifacts/model.tar.gz",
    role_arn="arn:aws:iam::943755222667:role/sagemaker-sudo-access",
    instance_type="ml.t2.medium",
)

model_builder.build(model_name="my-sklearn-model")

predictor = model_builder.deploy(
    endpoint_name="vimal-sklearn-endpoint",
    initial_instance_count=1,
)

[09/23/26 08:49:52] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=8512797;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=8512798;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#325\325]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    DEBUG    No ModelMetadata provided. ModelBuilder is not handling    ]8;id=8512805;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=8512806;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1389\1389]8;;\
                             MLflow model input                                                                    

[09/23/26 08:49:53] INFO     Cannot simulate policies for                                  ]8;id=8512813;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=8512814;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#422\422]8;;\
                             'arn:aws:iam::943755888667:role/sagemaker-sudo-access'                                
                             (access denied); permission verdict unknown.                                          

                    WARNING  Could not verify permissions for role                         ]8;id=8512820;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=8512821;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#657\657]8;;\
                             'arn:aws:iam::943755888667:role/sagemaker-sudo-access'                                
                             (caller lacks iam:SimulatePrincipalPolicy). Proceeding with                           
                             it. If the operation later fails with an access-denied error,                         
                             ensure the role has the required permissions for 'serving'                            
                             (see IamRoleResolver().get_required_actions('serving')) or                            
                             create a dedicated role via                                                           
                             IamRoleResolver().create_execution_role(role_type='serving').                         

[09/23/26 08:49:54] INFO     Cannot simulate policies for                                  ]8;id=8512826;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=8512827;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#422\422]8;;\
                             'arn:aws:iam::943755888667:role/sagemaker-sudo-access'                                
                             (access denied); permission verdict unknown.                                          

...

                    INFO     Creating endpoint-config with name vimal-sklearn-endpoint       ]8;id=8512877;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=8512878;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1093\1093]8;;\

                    INFO     Creating endpoint with name vimal-sklearn-endpoint              ]8;id=8512884;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=8512885;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1125\1125]8;;\

[09/23/26 08:50:15] WARNING  Failed to enable live logging: An error occurred                ]8;id=8512891;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=8512892;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#2844\2844]8;;\
                             (AccessDeniedException) when calling the FilterLogEvents                              
                             operation: User:                                                                      
                             arn:aws:sts::943755888667:assumed-role/sagemaker-sudo-access/Sa                       
                             geMaker is not authorized to perform: logs:FilterLogEvents on                         
                             resource:                                                                             
                             arn:aws:logs:ap-south-1:943755888667:log-group:/aws/sagemaker/E                       
                             ndpoints/vimal-sklearn-endpoint because no identity-based                             
                             policy allows the logs:FilterLogEvents action. Fallback to                            
                             default logging...                                                                    

Output()

[09/23/26 08:53:15] INFO     ✅ Deployment successful: Endpoint 'vimal-sklearn-endpoint'      ]8;id=8512898;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=8512899;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#3770\3770]8;;\
                             using None in SAGEMAKER_ENDPOINT mode (ARN:                                           
                             arn:aws:sagemaker:ap-south-1:943755888667:endpoint/vimal-sklearn                      
                             -endpoint)                                                                            

In [6]:
import tarfile

In [ ]:
with tarfile.open("sagemaker.tar.gz","w:gz") as tar:
    tar.add("deploy.py")
    tar.add("inference.py")
    tar.add("iris-model.pkl")
    tar.add("model_train.ipynb")
    tar.add("model.tar.gz")